In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


In [ ]:
# ======================
# 1. ПАРАМЕТРЫ
# ======================
DATA = Path("../data/house_prices.csv")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 0
TEST_SIZE = 0.2


In [ ]:
data = pd.read_csv(DATA)


In [ ]:
print("Number of rows:", data.shape[0])
print("Columns:", list(data.columns))


In [ ]:
data.info()
data.head()


In [ ]:
data.describe().round(2)


In [ ]:
# Распределение цены
plt.figure(figsize=(9, 5))
sns.histplot(data["price"], bins=40, kde=True, color="#21918c", edgecolor="white")
plt.title("Распределение цены квартир", fontsize=14, pad=15)
plt.xlabel("Цена", fontsize=12)
plt.ylabel("Количество", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(FIG / "houses_price_hist.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Цена vs площадь
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=data,
    x="area_sqm",
    y="price",
    hue="rooms",
    palette="viridis",
    alpha=0.55,
    s=35,
)
plt.title("Цена vs площадь (цвет = число комнат)", fontsize=14, pad=15)
plt.xlabel("Площадь, м²", fontsize=12)
plt.ylabel("Цена", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(FIG / "houses_price_vs_area.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Цена vs расстояние до центра
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=data,
    x="dist_center_km",
    y="price",
    alpha=0.4,
    s=25,
    color="#440154",
)
plt.title("Цена vs расстояние до центра", fontsize=14, pad=15)
plt.xlabel("Расстояние, км", fontsize=12)
plt.ylabel("Цена", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(FIG / "eda_price_vs_distance.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ======================
# 2. ПРИЗНАКИ И SPLIT
# ======================
feature_cols = ["area_sqm", "rooms", "floor", "year_built", "dist_center_km", "parking"]
target = "price"

X = data[feature_cols]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)
X_train.shape, X_test.shape


In [ ]:
# ======================
# 3. ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ МЕТРИК
# ======================
def eval_reg(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    print(f"{name:8s}  MAE={mae:8.1f}  RMSE={rmse:8.1f}  R2={r2:.3f}")
    return {"model": name, "MAE": mae, "RMSE": rmse, "R2": r2}


In [ ]:
# ======================
# 4. МОДЕЛИ
# ======================
models = {
    "lin": Pipeline([("sc", StandardScaler()), ("m", LinearRegression())]),
    "ridge": Pipeline([("sc", StandardScaler()), ("m", Ridge(1.0))]),
    "rf": RandomForestRegressor(n_estimators=300, max_depth=8, random_state=RANDOM_SEED),
}

rows = []
preds = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    preds[name] = pred
    rows.append(eval_reg(name, y_test, pred))

results = pd.DataFrame(rows)
results


In [ ]:
# ======================
# 5. ВИЗУАЛИЗАЦИЯ: ФАКТ vs ПРОГНОЗ (RF)
# ======================
pred_rf = preds["rf"]

plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=pred_rf, alpha=0.45, s=30, color="#31688e")
# Диагональ идеального прогноза
lims = [min(y_test.min(), pred_rf.min()), max(y_test.max(), pred_rf.max())]
plt.plot(lims, lims, "r--", lw=1.5, label="идеал y = ŷ")
plt.title("RandomForest: факт vs прогноз", fontsize=14, pad=15)
plt.xlabel("Факт (price)", fontsize=12)
plt.ylabel("Прогноз", fontsize=12)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(FIG / "houses_pred_vs_actual.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Остатки RF
residuals = y_test - pred_rf

plt.figure(figsize=(8, 5))
sns.scatterplot(x=pred_rf, y=residuals, alpha=0.4, s=25, color="#35b779")
plt.axhline(0, color="red", linestyle="--", lw=1.5)
plt.title("Остатки RandomForest (y − ŷ)", fontsize=14, pad=15)
plt.xlabel("Прогноз", fontsize=12)
plt.ylabel("Остаток", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(FIG / "houses_residuals.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
print(results.sort_values("RMSE").to_string(index=False))
